# Week 9 — Day 4
# Public Deployment: Going Live

## Melanoma Skin Lesion Classification — Streamlit on Hugging Face Spaces

This notebook continues directly from **Week 9 Day 3**. Day 3 wrapped the trained CNN in a Streamlit dashboard; Day 4 packages that dashboard for a public URL, validates the deployment contract, and tests the live application.


### What this notebook delivers

- A Colab-compatible deployment workflow.
- A reproducible `requirements.txt` and deployment folder.
- A public-hosting version of the Day 3 app with relative paths.
- Local smoke tests for files, metadata, preprocessing, and predictions.
- A Hugging Face Spaces deployment checklist.
- A live-URL test section for checking availability and prediction parity.


## Learning objectives

By the end of this notebook, you will be able to:

1. Explain why a fresh hosting environment needs an explicit `requirements.txt`.
2. Package a Streamlit app, serialized model, metadata, and preprocessing contract together.
3. Distinguish Colab paths from deployment-safe relative paths.
4. Deploy a Streamlit app to a free public host, with Hugging Face Spaces as the recommended route.
5. Test the public URL with edge cases and compare local and deployed predictions.
6. Record the public URL and reconcile local-vs-deployed differences.


## Project contract inherited from Day 3

| Item | Contract |
|---|---|
| Task | Melanoma skin-lesion classification |
| Classes | Benign / Malignant |
| Input | RGB image, resized to 128 × 128 |
| Preprocessing | OpenCV decode → BGR-to-RGB → resize → `float32 / 255` |
| Output | Malignant probability and thresholded class |
| Default threshold | 0.35, loaded from `model_metadata.json` |
| Model file | `melanoma_cnn.keras` |

The deployed app must preserve this contract. Changing preprocessing or dependency versions can change predictions.


In [1]:
!pip -q install streamlit tensorflow opencv-python-headless pillow requests
from pathlib import Path
import json, os, shutil, subprocess, sys, time, zipfile
import numpy as np
BASE_DIR = Path("/content")
PROJECT_DIR = BASE_DIR / "day4_public_deployment"
ARTIFACT_DIR = PROJECT_DIR / "deployment_artifacts"
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print("Project directory:", PROJECT_DIR)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 39.4 MB/s eta 0:00:00
Project directory: /content/day4_public_deployment


## Step 1 — Bring in the Day 3 deployment artifacts




In [2]:
from google.colab import files

print("Upload melanoma_cnn.keras and model_metadata.json (or a deployment_artifacts.zip).")
try:
    uploaded = files.upload()
    for name, data in uploaded.items():
        destination = PROJECT_DIR / name
        destination.write_bytes(data)
        print("Uploaded:", destination)
except Exception as exc:
    print("Upload skipped or unavailable:", exc)
for zip_path in list(PROJECT_DIR.glob("*.zip")):
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(PROJECT_DIR)
    print("Extracted:", zip_path.name)
for candidate in [PROJECT_DIR, PROJECT_DIR / "deployment_artifacts", PROJECT_DIR / "deployment_airtifacts"]:
    model = candidate / "melanoma_cnn.keras"
    metadata = candidate / "model_metadata.json"
    if model.exists() and metadata.exists():
        ARTIFACT_DIR.mkdir(exist_ok=True)
        if candidate != ARTIFACT_DIR:
            shutil.copy2(model, ARTIFACT_DIR / model.name)
            shutil.copy2(metadata, ARTIFACT_DIR / metadata.name)
        break

print("Model exists:", (ARTIFACT_DIR / "melanoma_cnn.keras").exists())
print("Metadata exists:", (ARTIFACT_DIR / "model_metadata.json").exists())


Upload melanoma_cnn.keras and model_metadata.json (or a deployment_artifacts.zip).


Saving deployment_artifacts (1).zip to deployment_artifacts (1).zip
Uploaded: /content/day4_public_deployment/deployment_artifacts (1).zip
Extracted: deployment_artifacts (1).zip
Model exists: True
Metadata exists: True


In [3]:
metadata_path = ARTIFACT_DIR / "model_metadata.json"
model_path = ARTIFACT_DIR / "melanoma_cnn.keras"

if metadata_path.exists():
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))

    print(json.dumps(metadata, indent=2))

    # Read image dimensions from the current metadata format
    image_height = int(metadata["input"]["height"])
    image_width = int(metadata["input"]["width"])
    image_size = (image_width, image_height)
    threshold = float(
        metadata["decision_rule"]["selected_threshold"]
    )
    class_mapping = metadata["classes"]
    labels = {
        int(class_id): class_name
        for class_name, class_id in class_mapping.items()
    }

    print("Serving contract:")
    print(f"- Image size: {image_width} x {image_height}")
    print(f"- Channels: {metadata['input']['channels']}")
    print(f"- Color format: {metadata['input']['color_format']}")
    print(f"- Preprocessing: {metadata['preprocessing']}")
    print(f"- Classes: {labels}")
    print(f"- Selected threshold: {threshold}")

else:
    print(
        "Metadata is missing. Upload the Day 2/Day 3 artifacts "
        "and rerun this cell."
    )



{
  "project": "Melanoma Skin Cancer \u2014 Benign vs Malignant",
  "sprint": "Sprint 4",
  "day": "Day 1",
  "model_format": "Keras .keras",
  "model_artifact": "melanoma_cnn.keras",
  "input": {
    "height": 128,
    "width": 128,
    "channels": 3,
    "color_format": "RGB"
  },
  "preprocessing": {
    "reader": "OpenCV",
    "color_conversion": "BGR to RGB",
    "resize": [
      128,
      128
    ],
    "dtype": "float32",
    "normalization": "pixel / 255.0"
  },
  "classes": {
    "Benign": 0,
    "Malignant": 1
  },
  "positive_class": "Malignant",
  "decision_rule": {
    "reference_threshold": 0.5,
    "selected_threshold": 0.35,
    "rule": "probability_malignant >= selected_threshold -> Malignant"
  },
  "week8_reference_metrics": {
    "threshold": 0.5,
    "accuracy": 0.8755,
    "precision_malignant": 0.9243,
    "recall_malignant": 0.818,
    "f1_malignant": 0.8679,
    "roc_auc": 0.9517,
    "pr_auc": 0.9445,
    "false_positives": 67,
    "false_negatives": 182
  }

## Step 2 — Create deployment-safe files

The Day 3 app used `/content/deployment_artifacts`, which is correct for Colab but not portable to a hosted repository. The following cell creates a deployment copy that resolves artifacts relative to `app.py`. This is the critical local-vs-deployed path fix.


In [4]:
import urllib.request
app_url = "https://raw.githubusercontent.com/LunaYahya/AI-Training/main/Week9/Day3/app.py"
app_text = urllib.request.urlopen(app_url).read().decode("utf-8")
app_text = app_text.replace(
    "ARTIFACT_DIR = Path('/content/deployment_artifacts')",
    "ARTIFACT_DIR = Path(__file__).resolve().parent / 'deployment_artifacts'")
(PROJECT_DIR / "app.py").write_text(app_text, encoding="utf-8")
requirements = """streamlit==1.36.0
tensorflow==2.20.0
numpy==2.1.3
pandas==2.2.3
opencv-python-headless==4.10.0.84
scikit-learn==1.6.1
joblib==1.6.0
pillow==11.3.0
"""
(PROJECT_DIR / "requirements.txt").write_text(requirements, encoding="utf-8")

readme = """# Melanoma Classifier — Public Demo

Streamlit deployment for Week 9 Day 4.

## Run locally
```bash
pip install -r requirements.txt
streamlit run app.py
```

## Model contract
Input: RGB image resized to 128 x 128 and scaled by 255.
Default decision threshold: 0.35 (loaded from model_metadata.json).

## Public URL
Replace this line with the live Space URL after deployment: `PUBLIC_URL = TODO`

Educational demonstration only; not medical advice or diagnosis.
"""
(PROJECT_DIR / "README.md").write_text(readme, encoding="utf-8")
print("Created:", PROJECT_DIR / "app.py")
print("Created:", PROJECT_DIR / "requirements.txt")


Created: /content/day4_public_deployment/app.py
Created: /content/day4_public_deployment/requirements.txt


## Step 3 — Inspect the deployment bundle

Every imported library must be represented in `requirements.txt`; every runtime file must be present in the repository. A missing model or metadata file causes a build or startup failure.


In [5]:
required = [PROJECT_DIR / "app.py", PROJECT_DIR / "requirements.txt", PROJECT_DIR / "README.md", ARTIFACT_DIR / "melanoma_cnn.keras", ARTIFACT_DIR / "model_metadata.json"]
for path in required:
    print(f"{path.relative_to(PROJECT_DIR)}: {'OK' if path.exists() else 'MISSING'}")

assert (PROJECT_DIR / "app.py").exists(), "app.py is missing"
assert (PROJECT_DIR / "requirements.txt").exists(), "requirements.txt is missing"
assert metadata_path.exists() and model_path.exists(), "Upload both model artifacts before deployment"
assert "streamlit" in (PROJECT_DIR / "requirements.txt").read_text()
assert "tensorflow" in (PROJECT_DIR / "requirements.txt").read_text()
assert "Path(__file__).resolve().parent" in (PROJECT_DIR / "app.py").read_text()
print("Deployment bundle validation passed.")


app.py: OK
requirements.txt: OK
README.md: OK
deployment_artifacts/melanoma_cnn.keras: OK
deployment_artifacts/model_metadata.json: OK
Deployment bundle validation passed.


## Step 4 — Test the model locally before deployment

This test repeats the same decoding and preprocessing used by the Streamlit app. It is a smoke test, not a clinical validation study. It checks that the model loads, accepts the expected shape, and returns a probability in the valid range.


In [6]:
import cv2
import tensorflow as tf

model = tf.keras.models.load_model(model_path)
labels = {int(v): k for k, v in metadata["classes"].items()}

def preprocess_image_bytes(image_bytes, image_size):
    buffer = np.frombuffer(image_bytes, dtype=np.uint8)
    image = cv2.imdecode(buffer, cv2.IMREAD_COLOR)
    if image is None:
        raise ValueError("The uploaded file is not a readable image.")
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, image_size, interpolation=cv2.INTER_AREA)
    image = image.astype(np.float32) / 255.0
    return np.expand_dims(image, axis=0)

print("Loaded model:", type(model).__name__)
print("Expected input shape:", model.input_shape)
print("Expected image size:", image_size)


Loaded model: Sequential
Expected input shape: (None, 128, 128, 3)
Expected image size: (128, 128)


In [7]:
from google.colab import files
print("Upload one JPG/PNG image for a local prediction test.")
try:
    test_upload = files.upload()
except Exception:
    test_upload = {}

local_result = None
for name, data in test_upload.items():
    x = preprocess_image_bytes(data, image_size)
    probability = float(np.asarray(model.predict(x, verbose=0)).squeeze())
    predicted_label = int(probability >= threshold)
    local_result = {"file": name, "malignant_probability": probability, "class": labels[predicted_label], "threshold": threshold}
    print(local_result)


Upload one JPG/PNG image for a local prediction test.


Saving 6312.jpg to 6312.jpg
{'file': '6312.jpg', 'malignant_probability': 0.04685937240719795, 'class': 'Benign', 'threshold': 0.35}


## Step 5 — Run the Streamlit app in Colab



In [8]:
!pkill -f "streamlit run" || true

import threading, subprocess, time, re
log_path = PROJECT_DIR / "streamlit.log"
log_file = open(log_path, "w")
process = subprocess.Popen(
    [sys.executable, "-m", "streamlit", "run", "app.py", "--server.port", "8503", "--server.address", "0.0.0.0", "--server.headless", "true"],
    cwd=PROJECT_DIR, stdout=log_file, stderr=subprocess.STDOUT
)
time.sleep(8)
print("Streamlit process:", process.poll() if process.poll() is not None else "running")
print(log_path.read_text(errors="ignore")[-2000:])


^C
Streamlit process: running


2026-09-16 14:35:21.123 Uvicorn server started on 0.0.0.0:8503

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8503
  Network URL: http://172.28.0.12:8503
  External URL: http://34.16.179.139:8503




In [9]:
from google.colab.output import eval_js
try:
    print(eval_js("google.colab.kernel.proxyPort(8503)"))
except Exception as exc:
    print("Open port 8503 through your Colab environment:", exc)


https://8503-gpu-t4-s-kkb-usw4b0-fmdgoxrvpe8p-b.us-west4-0.prod.colab.dev


## Step 6 — Deploy to Hugging Face Spaces

Hugging Face Spaces is the recommended free path for this Streamlit demo. The Space repository should contain exactly the deployment bundle:

```text
app.py
requirements.txt
README.md
deployment_artifacts/melanoma_cnn.keras
deployment_artifacts/model_metadata.json
```

### Manual deployment workflow

1. Create a new **Space** at [huggingface.co/new-space](https://huggingface.co/new-space).
2. Choose **Streamlit** as the SDK and select a public Space if appropriate for your project.
3. Clone the Space repository or use the browser upload interface.
4. Upload `app.py`, `requirements.txt`, `README.md`, and the complete `deployment_artifacts/` folder.
5. Wait for the build to complete. Read the **Logs** tab if the build fails.
6. Copy the public Space URL, for example `https://YOUR-USERNAME-melanoma-demo.hf.space`.

Do not place access tokens in notebook cells or commit them to GitHub. If using the command line, authenticate with the official Hugging Face CLI using a secret input mechanism.


In [10]:
zip_path = BASE_DIR / "melanoma-streamlit-space.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for path in PROJECT_DIR.rglob("*"):
        if path.is_file() and path.name != "streamlit.log":
            zf.write(path, path.relative_to(PROJECT_DIR))
print("Upload this bundle to the Space:", zip_path)
print("Bundle contents:")
with zipfile.ZipFile(zip_path) as zf:
    print("\n".join(zf.namelist()))


Upload this bundle to the Space: /content/melanoma-streamlit-space.zip
Bundle contents:
requirements.txt
deployment_manifest.json
app.py
melanoma_cnn.keras
model_metadata.json
deployment_artifacts (1).zip
README.md
deployment_artifacts/melanoma_cnn.keras
deployment_artifacts/model_metadata.json


## Step 7 — Test the live deployment

Paste the public URL below after the Space is running. Test the URL as a real user would: open the home page, upload several representative images, include an edge case such as a corrupted or unsupported file, and confirm the app remains understandable.

For prediction parity, use the same image bytes locally and in the deployed app. A visual match alone is not enough; compare the displayed malignant probability and class. Small numerical differences can occur because of runtime or hardware differences, but a large difference usually indicates a preprocessing, model-file, or dependency mismatch.


In [23]:
PUBLIC_URL = "https://ai-training-mipxqexmw7lfuh84hevpb2.streamlit.app/"
import requests
if PUBLIC_URL.strip():
    response = requests.get(PUBLIC_URL, timeout=30)
    print("Status:", response.status_code)
    print("Final URL:", response.url)
    print("Page contains Streamlit:", "streamlit" in response.text.lower())
    response.raise_for_status()
else:
    print("Set PUBLIC_URL after the Space is deployed, then rerun this cell.")


Status: 200
Final URL: https://ai-training-mipxqexmw7lfuh84hevpb2.streamlit.app/
Page contains Streamlit: True


## Summary

The application is now prepared for public deployment: the Day 3 Streamlit interface is preserved, the model-serving contract is explicit, deployment paths are portable, dependencies are pinned, and a verification process is documented. The final deliverable is not only a URL; it is a reproducible package that another user can build and test.
